# PARC2026 — π0.5 Dataset Cheap Ablation V1

`40_dataset_ablation_manifests.ipynb` で定義した V0/V1/V2 を、**同一π0.5 / 同一seed / 同一optimizer step** で短くscreeningします。

重要: このNotebookのtraining lossは候補絞り込み用です。**最終dataset選定はtraining lossだけで行わず、固定評価/simulator success evidenceが必要**です。

## Self-contained preflight
fresh Colab runtimeから直接実行できるよう、workspace・repo・dataset・manifest・training envをこのNotebook内で準備します。

In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
print('python:', sys.version)
print('platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)
ROOT = Path('/content/parc2026')
for p in [ROOT, ROOT/'vendor', ROOT/'cache', ROOT/'datasets', ROOT/'outputs']:
    p.mkdir(parents=True, exist_ok=True)
REPO = ROOT/'py_AI'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)
PI05_DIR = REPO/'examples/pi05_libero_finetune'
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())

## Python 3.10 / HF token
π0.5 trainingはLeRobot v0.4.4 + Python 3.10で固定します。PaliGemma access tokenは表示しません。

In [ ]:
if shutil.which('uv') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=True)
subprocess.run(['uv','python','install','3.10'], check=True)
PY310 = subprocess.check_output(['uv','python','find','3.10'], text=True).strip()
from getpass import getpass
if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
    except Exception:
        tok = None
    os.environ['HF_TOKEN'] = tok or getpass('HF token (PaliGemma access): ')
print('python3.10:', PY310)
print('HF_TOKEN: set (not displayed)')

## Training datasetを取得
運営combinedが無い場合は `lerobot/libero_plus` v3をrevision pinして取得します。v3はLeRobot v0.4.4のcodebase `v3.0` と互換です。meta/data/videosをColab一時領域へ置きます。

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.30','pyarrow>=16','pandas>=2'], check=True)
os.environ.setdefault('HF_HUB_DISABLE_XET','1')
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
PUBLIC_DATASET='lerobot/libero_plus'
PUBLIC_ROOT=ROOT/'datasets'/'public_libero_plus_v3_train'
ORGANIZER_ROOT=ROOT/'datasets'/'libero_combined_20hz'
configured=os.environ.get('PI05_DATASET_ROOT')
def train_ready(p): return (p/'meta'/'info.json').exists() and any(p.glob('data/**/*.parquet')) and (p/'videos').exists()
if configured and train_ready(Path(configured)):
    DATASET_ROOT=Path(configured); DATASET_ID=os.environ.get('PI05_DATASET_REPO_ID','local/libero_combined_20hz'); DATASET_REVISION=None
elif train_ready(ORGANIZER_ROOT):
    DATASET_ROOT=ORGANIZER_ROOT; DATASET_ID='local/libero_combined_20hz'; DATASET_REVISION=None
else:
    api=HfApi(); ds=api.dataset_info(PUBLIC_DATASET); DATASET_REVISION=ds.sha
    files=api.list_repo_files(PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION)
    required=[f for f in files if f.startswith('meta/') or (f.startswith('data/') and f.endswith('.parquet')) or (f.startswith('videos/') and f.endswith('.mp4'))]
    print('public revision:', DATASET_REVISION, 'files:', len(required))
    for i,filename in enumerate(required,1):
        if i==1 or i%20==0 or i==len(required): print(f'download {i}/{len(required)}: {filename}')
        hf_hub_download(repo_id=PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION, filename=filename, local_dir=str(PUBLIC_ROOT))
    DATASET_ROOT=PUBLIC_ROOT; DATASET_ID=PUBLIC_DATASET
print('dataset:', DATASET_ID, '@', DATASET_REVISION)
print('root:', DATASET_ROOT)

## Static metrics / manifestsを自動準備
同runtimeに40のoutputsが無くても、data parquetからStatic Analyzer → manifest generatorまで自動実行します。

In [ ]:
STATIC_OUT=ROOT/'outputs'/'static_quality_v1'
METRICS=STATIC_OUT/'episode_quality_metrics.csv'
if not METRICS.exists():
    subprocess.run([sys.executable,str(REPO/'tools/data/static_quality_analyzer.py'),'--root',str(DATASET_ROOT),'--out',str(STATIC_OUT),'--smooth-window','5','--robust-z-threshold','5.0'], check=True)
MANIFEST_OUT=ROOT/'outputs'/'dataset_ablation_manifests_v1'
if not (MANIFEST_OUT/'run_matrix.json').exists():
    cmd=[sys.executable,str(REPO/'tools/data/build_dataset_ablation_manifests.py'),'--metrics-csv',str(METRICS),'--out',str(MANIFEST_OUT),'--dataset-id',DATASET_ID,'--seed','20260830','--eval-per-task','2']
    if DATASET_REVISION: cmd += ['--dataset-revision',DATASET_REVISION]
    subprocess.run(cmd, check=True)
matrix=json.loads((MANIFEST_OUT/'run_matrix.json').read_text())
print('cheap order:', matrix['cheap_ablation_order'])
display(pd.DataFrame([{'variant':k,'episodes':v['episode_count'],'frames':v['frame_count']} for k,v in matrix['variants'].items()]))

## π0.5 training env setup
venv/model cache/checkpointは `/content/parc2026` 配下へ置きます。

In [ ]:
TRAIN_DATA_ROOT=ROOT/'cache'/'pi05-ablation-train'
LEROBOT_ROOT=ROOT/'vendor'/'lerobot-pi05-ablation'
TRAIN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
setup_env=os.environ.copy(); setup_env.update({'PYTHON':PY310,'DATA_ROOT':str(TRAIN_DATA_ROOT),'LEROBOT_ROOT':str(LEROBOT_ROOT)})
subprocess.run(['bash','scripts/setup_train.sh'], cwd=PI05_DIR, env=setup_env, check=True)
print('training env: ready')

## Cheap ablation設定
デフォルトは安全側の `BS=4 / GA=8 / 150 optimizer steps / seed=1000`。V1_INTEGRITY_ONLYは今回V0と同一なのでrun_matrixから自動skipします。

長時間runを誤って始めないよう `RUN_ABLATIONS=False` が初期値です。設定確認後にTrueへ変更します。

In [ ]:
ABLATION_BS=4
ABLATION_GA=8
ABLATION_STEPS=150
ABLATION_SEED=1000
RUN_ABLATIONS=False
print('effective batch:', ABLATION_BS*ABLATION_GA)
print('variants:', matrix['cheap_ablation_order'])
print('RUN_ABLATIONS:', RUN_ABLATIONS)

In [ ]:
RESULTS_DIR=ROOT/'outputs'/'pi05_dataset_ablation_v1'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
result_rows=[]
if RUN_ABLATIONS:
    for variant in matrix['cheap_ablation_order']:
        manifest=MANIFEST_OUT/f'{variant}.json'
        run_name='colab_data_'+variant.lower()
        env=os.environ.copy()
        env.update({
            'DATA_ROOT':str(TRAIN_DATA_ROOT), 'LEROBOT_ROOT':str(LEROBOT_ROOT),
            'ABLATION_MANIFEST':str(manifest), 'PI05_DATASET_ROOT':str(DATASET_ROOT),
            'PI05_DATASET_REPO_ID':DATASET_ID, 'PI05_VIDEO_BACKEND':'pyav',
            'ABLATION_BS':str(ABLATION_BS), 'ABLATION_GA':str(ABLATION_GA),
            'ABLATION_STEPS':str(ABLATION_STEPS), 'ABLATION_SEED':str(ABLATION_SEED),
            'RUN_NAME':run_name, 'HF_TOKEN':os.environ['HF_TOKEN'],
        })
        print('\n=== RUN', variant, '===')
        subprocess.run(['bash','-lc','source env_train.sh && bash scripts/cheap_ablation_pi05.sh'], cwd=PI05_DIR, env=env, check=True)
        s=TRAIN_DATA_ROOT/'pi05-ft-outputs'/run_name/'cheap_ablation_summary.json'
        row=json.loads(s.read_text()); result_rows.append(row)
        (RESULTS_DIR/f'{variant}.json').write_text(json.dumps(row,indent=2)+'\n')
    (RESULTS_DIR/'comparison.json').write_text(json.dumps(result_rows,indent=2)+'\n')
else:
    print('skip: set RUN_ABLATIONS=True after reviewing the settings above')

## 結果比較
lossはscreening参考値に留め、wall time / peak VRAM / manifest hashも同時に固定します。

In [ ]:
if (RESULTS_DIR/'comparison.json').exists():
    result_rows=json.loads((RESULTS_DIR/'comparison.json').read_text())
    cols=['variant','episode_count','frame_count','optimizer_steps','effective_batch','wall_sec','peak_vram_mib','final_logged_loss_best_effort','last20_logged_loss_mean_best_effort']
    display(pd.DataFrame(result_rows)[cols].sort_values('last20_logged_loss_mean_best_effort'))
    print('SCREENING COMPLETE — do not promote from loss alone')
else:
    print('no comparison.json yet')

## Exit criteria / 次
4 variantのsummaryが揃ったら、lossだけでwinnerを決めず **V0 + 有望なclean/balance候補** を固定local evalへ送ります。その後にmodel comparison Laneと同期します。